In [14]:
from pathlib import Path
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt 

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score
)

import json

RANDOM_STATE = 42
N_INIT = 10

In [7]:
DATA = {
    "ds1": "./data/S07-hw-dataset-01.csv",
    "ds2": "./data/S07-hw-dataset-02.csv",
    "ds3": "./data/S07-hw-dataset-03.csv"
}

In [34]:
dataframes = {k: pd.read_csv(d) for k, d in DATA.items()}

def take_info(df):
    return {
        "samples": len(df),
        "head": df.head(),
        "describe": df.describe(),
        "missings": int(df.isna().sum().sum()),
        "dtypes": df.dtypes.astype(str).to_dict()
    }
    
summarized_info = {k: take_info(df) for k, df in dataframes.items()}

summarized_info

{'ds1': {'samples': 12000,
  'head':    sample_id        f01        f02       f03         f04        f05  \
  0          0  -0.536647 -69.812900 -0.002657   71.743147 -11.396498   
  1          1  15.230731  52.727216 -1.273634 -104.123302  11.589643   
  2          2  18.542693  77.317150 -1.321686 -111.946636  10.254346   
  3          3 -12.538905 -41.709458  0.146474   16.322124   1.391137   
  4          4  -6.903056  61.833444 -0.022466  -42.631335   3.107154   
  
           f06        f07       f08  
  0 -12.291287  -6.836847 -0.504094  
  1  34.316967 -49.468873  0.390356  
  2  25.892951  44.595250  0.325893  
  3   2.014316 -39.930582  0.139297  
  4  -5.471054   7.001149  0.131213  ,
  'describe':          sample_id           f01           f02           f03           f04  \
  count  12000.00000  12000.000000  12000.000000  12000.000000  12000.000000   
  mean    5999.50000     -2.424716     19.107804     -0.222063     -8.284501   
  std     3464.24595     11.014315     60.7

In [11]:
preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [13]:
def calculate_cluster_metrics(X, labels):
    
    unique = np.unique(labels)
    
    if len(unique) < 2: 
        return None 
    
    try: 
        sil = float(silhouette_score(X, labels))
    except Exception:
        sil = None
        
    try: 
        db = float(davies_bouldin_score(X, labels))
    except Exception:
        db = None
    
    try: 
        ch = float(calinski_harabasz_score(X, labels))
    except Exception:
        ch = None
    
    return {
        "silhouette": sil,
        "davis_bouldin": db,
        "calinski_harabasz": ch
    }

In [16]:
all_metrics = {}
best_cfgs = {}
labels_store = {}

for name, dataframe in dataframes.items():
    X = dataframe.drop(columns=["sample_id"])
    preprocessed_X = preprocessor.fit_transform(X)
    
    sil_scr = []
    k_arr = range(2,15)
    
    for k in k_arr: 
        kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=N_INIT)
        labels = kmeans.fit_predict(preprocessed_X)
        sil_scr.append(silhouette_score(preprocessed_X, labels))
        
    best_k = k_arr[np.argmax(sil_scr)]

    plt.figure()
    plt.plot(k_arr, sil_scr, marker="o")
    plt.xlabel("k")
    plt.ylabel("Silhouette")
    plt.title(f"{name}: silhouette vs k")
    plt.savefig(f"artifacts/figures/{name}_silhouette_vs_k.png")
    plt.close()
    
    kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=N_INIT)
    labels = kmeans.fit_predict(preprocessed_X)

    all_metrics.setdefault(name, {})["KMeans"] = calculate_cluster_metrics(preprocessed_X, labels)
    best_cfgs[name] = {"method": "KMeans", "k_value": best_k}
    
    labels_store[name] = pd.DataFrame({
        "sample_id": dataframe["sample_id"], 
        "cluster_label": labels
    })


In [41]:
eps_arr = [0.5, 1, 1.5, 2]
db_sil_scr = []
    
for eps in eps_arr: 
    db = DBSCAN(eps=eps, min_samples=6)
    labels = db.fit_predict(preprocessed_X)  
    
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    if n_clusters > 1 and (labels != -1).sum() > 10:  
        score = silhouette_score(preprocessed_X, labels)
        db_sil_scr.append(score)
    else:
        db_sil_scr.append(-1) 
        

valid_indices = [i for i, score in enumerate(db_sil_scr) if score > -1]
if valid_indices:
    best_idx = valid_indices[np.argmax([db_sil_scr[i] for i in valid_indices])]
    best_eps = eps_arr[best_idx]
    print(f"Лучший eps: {best_eps} (silhouette = {db_sil_scr[best_idx]:.3f})")
else:
    print("Не найдено нужных параметров")
    
    
for name, df in dataframes.items():
    print(f"\n=== Анализ датасета: {name} ===")
            
    X = df.drop(columns=['sample_id'])
    preprocessed_X = preprocessor.fit_transform(X)
    
    db = DBSCAN(eps=best_eps, min_samples=6)
    labels = db.fit_predict(preprocessed_X)
    
    noise_ratio = (labels == -1).mean()
    mask = labels != -1
    
    if mask.sum() > 10:
        metrics = calculate_cluster_metrics(preprocessed_X[mask], labels[mask])
        metrics["noise_ratio"] = float(noise_ratio)
        all_metrics[name]["DBSCAN"] = metrics
        
print(all_metrics)

Лучший eps: 0.5 (silhouette = 0.101)

=== Анализ датасета: ds1 ===

=== Анализ датасета: ds2 ===

=== Анализ датасета: ds3 ===
{'ds1': {'KMeans': {'silhouette': 0.5216395622404243, 'davis_bouldin': 0.6853295219054456, 'calinski_harabasz': 11786.954622671532}, 'DBSCAN': {'silhouette': 0.28384832285882783, 'davis_bouldin': 0.98533275897006, 'calinski_harabasz': 4939.694863362494, 'noise_ratio': 0.03891666666666667}}, 'ds2': {'KMeans': {'silhouette': 0.3068610017701601, 'davis_bouldin': 1.3234721699867644, 'calinski_harabasz': 3573.3933329348392}, 'DBSCAN': {'silhouette': 0.02224288642809455, 'davis_bouldin': 0.758111561911708, 'calinski_harabasz': 41.597486627044596, 'noise_ratio': 0.052375}}, 'ds3': {'KMeans': {'silhouette': 0.31554470037825183, 'davis_bouldin': 1.1577256320598661, 'calinski_harabasz': 6957.162639510167}, 'DBSCAN': {'silhouette': 0.17011838160369872, 'davis_bouldin': 0.7354022123604214, 'calinski_harabasz': 16.287515342465277, 'noise_ratio': 0.026}}}


In [26]:

#графики PCA 
for name, df in dataframes.items():
    X = df.drop(columns=["sample_id"])
    preprocessed_X = preprocessor.fit_transform(X)
    
    labels = labels_store[name]["cluster_label"]
    
    
    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    X2 = pca.fit_transform(preprocessed_X)
    
    plt.figure()
    plt.scatter(X2[:,0], X2[:,1], c=labels, s=10)
    plt.title(f"{name}: PCA(2D)")
    plt.savefig(f"artifacts/figures/{name}_PCA2D.png")
    plt.close()


# Проверка разных RANDOM STATE
X = dataframes["ds2"].drop(columns=["sample_id"])
preprocessed_X = preprocessor.fit_transform(X)

labels_list = []

for rs in range(5):
    km = KMeans(n_clusters=best_cfgs["ds2"]["k_value"], random_state=rs, n_init=10)
    labels_list.append(km.fit_predict(preprocessed_X))

ari_scores = [
    adjusted_rand_score(labels_list[0], labels_list[i])
    for i in range(1, 5)
]

stability_mean_ari = float(np.mean(ari_scores))

stability_mean_ari

0.999250093753161

In [42]:
with open("artifacts/metrics_summary.json", "w") as f:
    json.dump(all_metrics, f, indent=2)


In [33]:
for name, df_lab in labels_store.items():
    df_lab.to_csv(f"./artifacts/labels/labels_hw07_{name}.csv",
        index=False
    )

In [27]:
with open("artifacts/best_configs.json", "w") as f:
    json.dump(best_cfgs, f, indent=2)